# Lesson 4: What RAG really costs: chunk size, k, and the long-context alternative

*Module 2 · about 12 minutes · API key needed for the k sweep (chunking and retrieval run offline)*

Retrieval-augmented generation (RAG) is the standard way to let a model answer from your own documents. You split the documents into chunks, find the chunks most relevant to a question, and paste them into the prompt. It works well. It also has a cost that's easy to miss: every chunk you retrieve is billed as input tokens, on every single question.

Most RAG pipelines retrieve a fixed number of chunks, `k`, and very often `k` is just whatever the framework's example used. In this lesson we'll build a tiny RAG pipeline you can see all the way through, and then measure how chunk size and `k` change the bill and whether the answer stays right.

By the end you should be able to:

1. Explain where RAG spends money, and why the query-time input usually dominates.
2. See how chunk size changes the number of tokens you send per question.
3. Find the smallest `k` that still answers correctly, with the cost printed next to it.
4. Compare RAG with simply putting the whole document in the prompt, and reason about when the extra cost is worth it.


### Where the money goes in RAG

There are three places:

1. **Ingest (one-off):** splitting documents and computing an *embedding* for each chunk. An embedding is a list of numbers that represents what a piece of text is about, so that similar texts get similar numbers. Embedding models are very cheap per token, and you only pay once per document version.
2. **Storage:** keeping those vectors in a database. Usually small.
3. **Query time (every question):** embedding the question (tiny), then sending the top `k` chunks to the model as input. **This is the big one.** If each chunk is 500 tokens and `k` is 20, that's 10,000 input tokens per question before the model has written anything.

So the two dials that matter most for cost are chunk size and `k`. They matter for quality too: too little context and the answer is missing; too much and the model has to find the relevant sentence among lots of irrelevant ones, which can make answers worse as well as more expensive.

To keep everything visible we won't use a vector database or a real embedding model. We'll use a simple word-overlap similarity instead. It's cruder than real embeddings, but the cost mechanics are exactly the same.


### How these notebooks work

Run the cells in order, top to bottom. Before each code cell there's a short explanation of what it does and what to look at in the output. After the important ones there's a note on how to read what you got. Your numbers won't match mine exactly, because models are non-deterministic and prices change, so the notes describe what to look for rather than quoting fixed values.

A few conventions:

- **In class:** notes are cues for when we run this together. If you're working alone, just read them as a prompt to stop and think.
- Every notebook that spends money ends with a **ledger**: one row per API call and the total you spent.
- The **Check yourself** questions at the end have answers hidden under a click. Try them before you look.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, vendor_tokens, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
LIVE = cfg.live


  Provider : anthropic
  floor    : claude-haiku-4-5
  mid      : claude-sonnet-5
  frontier : claude-opus-5
  Cache    : explicit cache_control; read/write are separate buckets.
Switch with LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env
  Rate card: verified 5 Sep 2026 — re-check before presenting.


That cell reads your `.env`, picks OpenAI or Anthropic depending on which key it finds, and prints the three model tiers the notebook will use (floor, mid, frontier).

If the banner names a provider, the live cells will make real calls. Every lesson costs cents, not dollars. If it says `offline`, all the arithmetic still runs, but cells that need a model's answer print a placeholder and tell you they can't draw a conclusion. You can read an offline run, but it's no substitute for a live one in the caching, compression, and routing lessons.

To switch vendors, set `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and run the cell again.


---
## 1. A small company handbook

Our document is a made-up operations handbook for a logistics company, about 6,000 tokens across 16 sections. Six sections are customer policy (refunds, tracking, delays, claims, escalation). The other ten are internal operations material (warehouse safety, shift scheduling, customs, and so on), which is the kind of text that sits in the same knowledge base but is irrelevant to customer questions.

One section is there on purpose as a **distractor**: wholesale refunds follow a different rule, with a different dollar threshold. That's realistic. Real knowledge bases are full of near-misses like this.


In [2]:
import math, re, textwrap
from collections import Counter

POLICY_SECTIONS = {
    "Refunds (consumer)":
        "Refunds under $500 are auto-approved when a shipment is delayed more than 48 hours "
        "and the customer has fewer than three claims in the trailing twelve months. "
        "Claims above $500 require supervisor approval and a case note. "
        "A customer with three or more claims in that period is routed to manual review.",
    "Refunds (wholesale accounts)":
        "Wholesale accounts follow a separate refund schedule negotiated in each master agreement. "
        "As a default, wholesale credits under $2,000 are issued as account credit rather than cash, "
        "and disputes are handled by the account manager, not the support desk.",
    "Tracking":
        "Tracking numbers use the format NW-########. Customers can track shipments in the "
        "portal or by SMS. Tracking data refreshes every 30 minutes from carrier feeds.",
    "Delays":
        "A shipment is considered delayed when it exceeds the committed delivery window by "
        "more than 24 hours. Weather exclusions apply during declared severe weather events.",
    "Damage claims":
        "Damage claims must be filed within 14 days of delivery with photographic evidence. "
        "Claims filed after 14 days are rejected automatically unless a supervisor overrides.",
    "Escalation":
        "Escalate to a human agent when the customer requests it, when the claim exceeds "
        "$500, or when the customer has three or more open cases.",
}

# Ten sections of internal operations text: long, plausible, and irrelevant to customers.
OPS_TOPICS = {
    "Warehouse safety":    ["forklift", "aisle", "pallet", "helmet", "inspection"],
    "Shift scheduling":    ["rota", "overtime", "handover", "roster", "break"],
    "Seasonal staffing":   ["temporary", "peak", "onboarding", "agency", "induction"],
    "Customs and export":  ["tariff", "declaration", "broker", "commodity", "origin"],
    "Packaging standards": ["carton", "void fill", "labelling", "weight", "seal"],
    "Hazardous goods":     ["lithium", "placard", "segregation", "manifest", "UN number"],
    "Fleet maintenance":   ["tyre", "service interval", "telematics", "brake", "mileage"],
    "IT and access":       ["badge", "password", "VPN", "scanner", "handheld"],
    "Sustainability":      ["emissions", "route planning", "idling", "recycling", "electric van"],
    "Carrier contracts":   ["surcharge", "lane", "capacity", "rate card", "service level"],
}

def ops_section(topic, words, paragraphs=6):
    out = []
    for p in range(paragraphs):
        w = words[p % len(words)]
        out.append(
            f"This part of the {topic.lower()} guide covers {w} procedures for site {p + 1}. "
            f"Team leads review {w} records at the start of each week and log exceptions in the "
            f"operations tracker. Where a {w} issue affects more than one site, the regional "
            f"manager is informed and a corrective action is agreed within five working days. "
            f"New staff complete {w} training during their first month, and refresher sessions "
            f"run every quarter. Questions about {w} should go to the {topic.lower()} lead rather "
            f"than the support desk."
        )
    return "\n\n".join(out)

HANDBOOK = dict(POLICY_SECTIONS)
for topic, words in OPS_TOPICS.items():
    HANDBOOK[topic] = ops_section(topic, words)

print(f"{len(HANDBOOK)} sections, {sum(ntok(t) for t in HANDBOOK.values()):,} tokens in total")


16 sections, 5,939 tokens in total


---
## 2. Chunking

Before we can retrieve anything we have to cut the handbook into chunks. We'll use **recursive splitting**, the simplest method that respects the structure of the text. It tries to split on paragraph breaks first. If a paragraph is still too big, it splits that paragraph on sentence ends, and only as a last resort on spaces. Pieces are then packed back together up to the size limit. Each section is chunked on its own, so a chunk never straddles two unrelated sections.

Why this method? Published comparisons in 2026 keep finding that it does as well as, or better than, more elaborate methods:

| Strategy | End-to-end answer accuracy | Model calls at ingest |
|---|---|---|
| Recursive splitting, ~512 tokens | 69% (best) | none |
| Fixed-size windows, 512 tokens | 67% | none |
| Semantic chunking | 54% (despite 91.9% retrieval recall) | an embedding call per sentence |

*(FloTorch 2026 benchmark across 50 papers, alongside results from NVIDIA and Chroma Research.)*

The semantic chunker found the right passages more often but produced tiny fragments (about 43 tokens on average) that didn't carry enough context for the model to answer. Finding the right text and answering well are different things, and only the second one matters to your users.

**Overlap**, where consecutive chunks repeat a little text so that sentences on the boundary aren't cut in half, is something to test rather than assume. One 2026 analysis found it made no measurable difference in its setup while increasing indexing cost. We leave it out here.

The cell below chunks the handbook at four sizes and shows what that means per question: how many chunks there are, how big they are, and how many tokens `k = 3` would send to the model.


In [3]:
def split_recursive(text, size, seps=("\n\n", ". ", " ")):
    """Split on paragraphs, then sentences, then words; pack pieces up to `size` tokens."""
    if ntok(text) <= size or not seps:
        return [text]
    sep, rest = seps[0], seps[1:]
    if sep not in text:
        return split_recursive(text, size, rest)
    pieces = [p + sep for p in text.split(sep)]
    pieces[-1] = pieces[-1][: -len(sep)]
    chunks, cur = [], ""
    for p in pieces:
        if ntok(cur + p) <= size:
            cur += p
            continue
        if cur:
            chunks.append(cur)
        if ntok(p) > size:
            chunks.extend(split_recursive(p, size, rest))
            cur = ""
        else:
            cur = p
    if cur:
        chunks.append(cur)
    return [c.strip() for c in chunks if c.strip()]

rows = []
for size in [128, 256, 512, 1024]:
    chunks = [c for text in HANDBOOK.values() for c in split_recursive(text, size)]
    toks = [ntok(c) for c in chunks]
    avg = sum(toks) / len(toks)
    rows.append(dict(chunk_size=size, chunks=len(chunks), avg_tokens=round(avg),
                     largest=max(toks), tokens_sent_at_k3=round(3 * avg)))
show(pd.DataFrame(rows))


,chunk_size,chunks,avg_tokens,largest,tokens_sent_at_k3
0,128,66,90,99,270
1,256,36,165,193,495
2,512,26,228,480,685
3,1024,16,371,576,1114


**Reading the output.** Bigger chunks mean fewer chunks to embed and store, which makes ingest cheaper. But every retrieved chunk is bigger too, so each question sends more text to the model, and most of that text is whatever happened to sit next to the sentence you needed. Smaller chunks are more precise but can split an answer across two chunks.

Notice that the largest chunk is often well under the limit. The splitter only merges whole paragraphs, and it won't merge across sections.

Sensible starting points, which you should then test on your own questions:

- Short factual questions (names, dates, thresholds): **256–512 tokens**
- Questions that need explanation or comparison: **512–1,024 tokens**
- Overlap: start at none or about 10–15%, and keep it only if your eval says it helps.


---
## 3. Retrieval

We'll use 256-token chunks for the rest of the notebook, which gives us a few dozen chunks. To find the relevant ones we score every chunk against the question with **cosine similarity** over word counts: roughly, how much vocabulary the two texts share, adjusted for length. A real system would compare embedding vectors instead, which also catches synonyms. The logic of "score everything, take the top k" is the same.

The question we'll ask throughout: *What is the refund threshold and how many prior claims disqualify a customer?* The answer is in the consumer refunds section. Look at which sections come back in the top five, and where the wholesale distractor lands.


In [4]:
STOP = set("the a an and or of to in on for with is are be by when than more has have what how "
           "many which who this that it its as at from per each".split())

def vec(text):
    return Counter(w for w in re.findall(r"[a-z0-9$]+", text.lower()) if w not in STOP and len(w) > 1)

def cos(a, b):
    num = sum(a[w] * b[w] for w in set(a) & set(b))
    den = math.sqrt(sum(v * v for v in a.values())) * math.sqrt(sum(v * v for v in b.values()))
    return num / den if den else 0.0

CHUNK_SIZE = 256
CHUNKS = [(title, c) for title, text in HANDBOOK.items() for c in split_recursive(text, CHUNK_SIZE)]
CVECS = [vec(c) for _, c in CHUNKS]

def retrieve(query, k):
    qv = vec(query)
    order = sorted(range(len(CHUNKS)), key=lambda i: cos(qv, CVECS[i]), reverse=True)
    return [(CHUNKS[i][0], CHUNKS[i][1], cos(qv, CVECS[i])) for i in order[:k]]

QUESTION = "What is the refund threshold and how many prior claims disqualify a customer?"
print(f"{len(CHUNKS)} chunks of up to {CHUNK_SIZE} tokens\n")
print("top 5 for the question:")
for title, text, score in retrieve(QUESTION, 5):
    print(f"  {score:.3f}  {title:<30} {textwrap.shorten(text, 60)}")


36 chunks of up to 256 tokens

top 5 for the question:
  0.311  Refunds (consumer)             Refunds under $500 are auto-approved when a shipment [...]
  0.218  Escalation                     Escalate to a human agent when the customer requests [...]
  0.154  Damage claims                  Damage claims must be filed within 14 days of delivery [...]
  0.073  Refunds (wholesale accounts)   Wholesale accounts follow a separate refund schedule [...]
  0.000  Tracking                       Tracking numbers use the format NW-########. Customers [...]


The consumer refunds chunk should come first. After it come sections that share words like "claims" and "\$500" (escalation, damage claims) and then the wholesale refunds section, which shares "refund" but has the wrong threshold. Past the first few, the scores drop to zero: those chunks share no vocabulary with the question at all. With a large `k`, you'd still send them.


---
## 4. Sweeping k

Now the main experiment. We ask the same question with `k` = 20, 10, 5, 3, and 1, send the retrieved chunks to the model each time, and check two things:

- **Cost:** the billed input tokens and the price of the call.
- **Correctness:** we call an answer *grounded* if it gives both facts, the \$500 threshold and the three-claim limit. (Again a simple pattern check, not a judgement of style.)

Look for the smallest `k` that is still grounded, and how much cheaper it is than `k = 20`. Also look at whether the answers at large `k` mention the wholesale rule. More context isn't only more expensive, it's more for the model to sort through.

> **In class:** don't quote the token saving without the grounded column next to it.


In [5]:
def grounded(answer):
    return bool(re.search(r"\$?\b500\b", answer)) and bool(re.search(r"\b(3|three)\b", answer, re.I))

results = []
for k in [20, 10, 5, 3, 1]:
    ctx = "\n\n".join(text for _, text, _ in retrieve(QUESTION, k))
    r = complete(
        f"CONTEXT:\n{ctx}\n\nQUESTION: {QUESTION}",
        system="Answer only from the provided context. If it is not there, say so. Two sentences at most.",
        model=MODELS.mid,
        max_tokens=200,
        label=f"k={k}",
    )
    ok = None if r.fallback else grounded(r.text)
    results.append(dict(k=k, input_tokens=r.fresh_input + r.cache_read + r.cache_write,
                        cost=r.usd, grounded=ok,
                        mentions_wholesale=None if r.fallback else "wholesale" in r.text.lower()))
    print(f"   k={k:<3} grounded={ok}   {r.text[:90]!r}")

df = pd.DataFrame(results)
df["monthly_at_1M_queries"] = df.cost * 1_000_000
show(df.style.format({"cost": "${:,.6f}", "monthly_at_1M_queries": "${:,.0f}"}))

if df.grounded.isna().all():
    print("\nNo live answers, so we can't say which k is good enough. The token counts above are still real.")
else:
    good = df[df.grounded == True]
    top = df.iloc[0]
    if good.empty:
        print("\nNo k produced a grounded answer. Check the retrieved chunks before tuning anything else.")
    else:
        best = good.sort_values("k").iloc[0]
        print(f"\nSmallest grounded k: {int(best.k)}. Compared with k={int(top.k)}: "
              f"{1 - best.cost / top.cost:.0%} cheaper per query "
              f"({usd(top.monthly_at_1M_queries)} -> {usd(best.monthly_at_1M_queries)} a month at 1M queries).")


k=20                                          $0.009656   in=4483    out=69     cw=0       cr=0       
   k=20  grounded=True   'The refund threshold for auto-approval is under $500 (for shipments delayed more than 48 h'


k=10                                          $0.003976   in=1618    out=74     cw=0       cr=0       
   k=10  grounded=True   'Refunds under $500 are auto-approved when a shipment is delayed more than 48 hours and the'


k=5                                           $0.001652   in=396     out=86     cw=0       cr=0       
   k=5   grounded=True   'Refunds under $500 are auto-approved when a shipment is delayed more than 48 hours, provid'


k=3                                           $0.001350   in=260     out=83     cw=0       cr=0       
   k=3   grounded=True   'Refunds under $500 are auto-approved (delayed shipment >48 hours), while claims of $500 or'


k=1                                           $0.000866   in=163     out=54     cw=0       cr=0       
   k=1   grounded=True   'The auto-approval threshold is refunds under $500, and having three or more claims in the '


,k,input_tokens,cost,grounded,mentions_wholesale,monthly_at_1M_queries
0,20,4483,$0.009656,True,False,"$9,656"
1,10,1618,$0.003976,True,False,"$3,976"
2,5,396,$0.001652,True,False,"$1,652"
3,3,260,$0.001350,True,False,"$1,350"
4,1,163,$0.000866,True,False,$866



Smallest grounded k: 1. Compared with k=20: 91% cheaper per query ($9,656.00 -> $866.00 a month at 1M queries).


**Reading the output.** On this corpus a very small `k` is enough, because the answer lives in a single chunk and our retrieval ranks it first. `k = 1` may still be grounded here. In production you usually want a little slack, because retrieval isn't always this reliable: some questions need two chunks, and the right chunk isn't always ranked first.

A common pattern is to **retrieve wide and then rerank**: pull, say, 20–50 candidates cheaply, use a reranker (a small model that scores question–chunk pairs more carefully than vector similarity) to reorder them, and send only the top 3–5 to the expensive model. You pay the reranker a little so you can pay the generator much less.


---
## 5. RAG versus putting everything in the prompt

Modern models accept very long prompts, so why retrieve at all? Why not paste the whole handbook in and let the model find the answer?

Sometimes that's the right call. A 2026 study, "The Token Tax of Epistemic Accuracy" (arXiv 2606.20898), compared the two approaches on a manufacturing-safety benchmark. Putting everything in the prompt was more accurate (73.1% correct against 65.4% for semantic RAG), but each question cost about **26 times** as many tokens.

So it's a trade: better answers for a lot more money. The cell below measures that multiple on our handbook. It will be smaller than 26× because our handbook is small. Real knowledge bases are far bigger, and the multiple grows with them. Prompt caching (Lesson 2) can soften it if the same large document is sent over and over, but it doesn't make it go away.


In [6]:
EVERYTHING = "\n\n".join(f"## {t}\n{txt}" for t, txt in HANDBOOK.items())
r = complete(
    f"CONTEXT:\n{EVERYTHING}\n\nQUESTION: {QUESTION}",
    system="Answer only from the provided context. If it is not there, say so. Two sentences at most.",
    model=MODELS.mid,
    max_tokens=200,
    label="whole handbook in the prompt",
)
c_long = r.usd
c_rag = df[df.k == 3].cost.iloc[0]
print(f"\nwhole handbook : {r.fresh_input + r.cache_read + r.cache_write:>6,} input tokens   {usd(c_long)} per question")
print(f"RAG, k=3       : {int(df[df.k == 3].input_tokens.iloc[0]):>6,} input tokens   {usd(c_rag)} per question")
print(f"multiple       : {c_long / c_rag:.1f}x")
if not r.fallback:
    print(f"grounded       : {grounded(r.text)}")


whole handbook in the prompt                    $0.0192   in=9224    out=78     cw=0       cr=0       

whole handbook :  9,224 input tokens   $0.0192 per question
RAG, k=3       :    260 input tokens   $0.001350 per question
multiple       : 14.2x
grounded       : True


Which to choose depends on what a wrong answer costs you. For an internal FAQ bot, a few points of accuracy probably aren't worth 10–26× the spend. For a question where a wrong answer means a compliance breach or a safety incident, they might be. Either way, make it a deliberate decision per use case, write down why, and revisit it when prices or models change.


In [7]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.0367


,label,model,input,output,cache_write,cache_read,usd,note
0,k=20,claude-sonnet-5,4483,69,0,0,0.009656,
1,k=10,claude-sonnet-5,1618,74,0,0,0.003976,
2,k=5,claude-sonnet-5,396,86,0,0,0.001652,
3,k=3,claude-sonnet-5,260,83,0,0,0.001350,
4,k=1,claude-sonnet-5,163,54,0,0,0.000866,
5,whole handbook in the prompt,claude-sonnet-5,9224,78,0,0,0.019228,


---
## What to take away

- In RAG, the recurring cost is the retrieved context you send on every query. Ingest and storage are usually small by comparison.
- Start with recursive splitting at around 512 tokens. It's cheap, simple, and held up best in the 2026 comparisons.
- Tune `k` with an eval, not a default. Find the smallest `k` that stays grounded, and leave a little slack.
- Retrieve wide, rerank, and send only a few chunks to the expensive model.
- Putting the whole corpus in the prompt can be more accurate, at a large multiple of the cost. Decide per use case, based on what an error costs.


### Check yourself

**1. Your chunks average 600 tokens and your pipeline uses k = 12. You handle 400,000 questions a month on a model charging \$2 per million input tokens. What does the retrieved context alone cost per month? What if k were 4?**

<details><summary>Show answer</summary>

600 × 12 = 7,200 tokens per question × 400,000 = 2.88 billion tokens = **\$5,760 a month**. At k = 4 it's a third of that, **\$1,920**, provided the eval shows answers still hold.

</details>

**2. A semantic chunker gives better retrieval recall than recursive splitting, yet worse final answers. How can both be true?**

<details><summary>Show answer</summary>

Recall measures whether the right text was *found*. Answer accuracy measures whether the model could *use* it. Very small semantic chunks can contain the right sentence but not the context around it that the model needs, so it's found but not usable.

</details>

**3. Why might answers at k = 20 be worse than at k = 3, not just more expensive?**

<details><summary>Show answer</summary>

The extra chunks are mostly irrelevant, and some are near-misses, like the wholesale refund rule here. The model has to pick the right fact out of more distractions, and it sometimes mixes in the wrong one.

</details>


### Try it on your own work

For one RAG feature, log `k` and the input tokens of each query for a week. Then write 20 real questions with the facts a correct answer must include, and run them at `k` = 20, 10, 5, and 3. Ship the smallest `k` that holds the score, plus one or two for safety.
